# Soft Cosine Similarity

**Medium** &nbsp;·&nbsp; TensorTonic &nbsp;·&nbsp; `NLP`

Plain cosine similarity assumes every feature is **independent** — the word
`car` and the word `automobile` are as unrelated as `car` and `banana`. Two
documents that say the same thing in different words score `0.0`.

Soft cosine similarity fixes that by taking a **feature similarity matrix** `S`,
where $s_{ij}$ is how similar feature `i` is to feature `j` (typically from word
embeddings), with $s_{ii} = 1$ on the diagonal.

$$\text{soft\_cosine}(a, b) =
\frac{\sum_{i,j} s_{ij}\, a_i b_j}
{\sqrt{\sum_{i,j} s_{ij}\, a_i a_j} \; \sqrt{\sum_{i,j} s_{ij}\, b_i b_j}}$$

Every pair of features contributes, weighted by how related they are. When
`S` is the identity matrix, every off-diagonal term vanishes and this collapses
back into ordinary cosine similarity.

---

**Example 1:**

```
Input:  a = [1, 0], b = [0, 1], S = [[1, 0.5], [0.5, 1]]
Output: 0.5
The two vectors share no features, but the features themselves are
50% similar. Plain cosine would say 0.0.
```

**Example 2:**

```
Input:  a = [1, 2, 3], b = [2, 4, 6], S = identity(3)
Output: 1.0    (S = I -> this is just cosine similarity)
```

---

**Hint 1:** the numerator is $a^\top S\, b$ — a matrix-vector product followed by
a dot product. No double loop needed.

**Hint 2:** the two denominator terms are the *same expression* with `b` swapped
for `a`. Write one helper that computes $x^\top S\, y$ and call it three times.

**Requirements:**

- input: 1-D NumPy arrays `a`, `b` of length `n`, and an `n x n` matrix `S`
- output: scalar float
- fully vectorized (no Python loops over features)
- return `0.0` if either denominator term is zero
- with `S = np.eye(n)` the result must equal plain cosine similarity

**Constraints:**

- `len(a) == len(b) == S.shape[0] == S.shape[1] <= 10^3`
- `S` is symmetric with `1.0` on the diagonal
- use only NumPy

---

You already have `dot_product`, `norm` and `cosine_similarity` from the earlier
problems. This one is the same three lines with `S` wedged into the middle.

### What is actually new here

The shape of the formula is identical to plain cosine — numerator over the
product of two square roots. The only change is that the dot product
$x \cdot y$ has become $x^\top S\, y$. So:

- $a \cdot b \;\rightarrow\; a^\top S\, b$
- $\|a\| = \sqrt{a \cdot a} \;\rightarrow\; \sqrt{a^\top S\, a}$

Write **one** helper, call it `bilinear(x, S, y)`, that returns $x^\top S\, y$,
and the rest of the function is three calls and a division. If you find yourself
writing a nested loop over `i` and `j`, stop — that is the $O(n^2)$ sum spelled
out longhand, and `x @ S @ y` is the same thing.

### The trap: what `S` has to be

Plain cosine can never fail on a non-zero vector, because $a \cdot a$ is a sum of
squares and cannot go negative. $a^\top S\, a$ **can**, and then `np.sqrt` hands
you a `nan` and a warning.

That happens when `S` is not **positive semi-definite**. Symmetric with ones on
the diagonal is not enough — try `S = [[1, 2], [2, 1]]` and `a = [1, -1]`, and
work out $a^\top S\, a$ by hand. You will get a negative number.

This is not a hypothetical. A similarity matrix built by dropping raw
`word2vec` cosines into a grid is symmetric with a unit diagonal and is very
often **not** PSD. The test cell checks it; decide what your function should do
about it. Two defensible answers — raise, or clamp — and one indefensible one,
which is returning `nan` silently.

### A second version worth writing

If `S` is PSD you can factor it as $S = L L^\top$ (`np.linalg.cholesky`). Then

$$x^\top S\, y = (L^\top x) \cdot (L^\top y)$$

so soft cosine similarity is *exactly* plain cosine similarity of the two
**transformed** vectors $L^\top a$ and $L^\top b$. Write
`soft_cosine_cholesky` and confirm it agrees with your direct version.

That is not a party trick. `L` does not depend on `a` or `b`, so you factor once
and then every later comparison is a plain cosine on pre-transformed vectors —
which is what lets a vector database serve soft cosine at the speed of ordinary
cosine.

### A third, for the shape you will actually use

`soft_cosine_matrix(a, matrix, S)` — one query against every **row** of a 2-D
array, no Python loop over the rows. `matrix @ S` gets you most of the way;
think about `axis=` and the `where=` guard for zero rows, same as before.

In [ ]:
import numpy as np


class Solution:
    def dot_product(self, x, y) -> float:
        if len(x) != len(y):
            raise ValueError("Vectors must have the same length")
        return float(np.dot(x, y))

    def norm(self, a):
        return np.sqrt(np.dot(a, a))

    def cosine_similarity(self, a, b) -> float:
        a_norm = self.norm(a)
        b_norm = self.norm(b)
        if a_norm == 0 or b_norm == 0:
            return 0.0
        return self.dot_product(a, b) / (a_norm * b_norm)

    # x^T S y - write this one first, everything else is three calls to it
    def bilinear(self, x, S, y) -> float:
        pass

    def soft_cosine_similarity(self, a, b, S) -> float:
        pass

    # optional: factor S = L L^T once, then it is plain cosine on L^T a and L^T b
    def soft_cosine_cholesky(self, a, b, S) -> float:
        pass

    # optional: one vector vs every row of a matrix -> array of similarities
    def soft_cosine_matrix(self, a, matrix, S):
        pass


In [ ]:
def check(got, want, tol=1e-9):
    """Compare one result against its expected value."""
    if got is None:
        return "not implemented"
    try:
        return "OK" if abs(got - want) < tol else f"WRONG got {got!r} want {want!r}"
    except TypeError:
        return f"WRONG got {got!r} (expected a number)"


sol = Solution()

I2 = np.eye(2)
I3 = np.eye(3)
S2 = np.array([[1.0, 0.5],
               [0.5, 1.0]])
S3 = np.array([[1.0, 0.9, 0.2],      # feature 0 and 1 are near-synonyms
               [0.9, 1.0, 0.2],
               [0.2, 0.2, 1.0]])

cases = [
    # (a, b, S, expected)
    ([1, 2, 3], [2, 4, 6], I3, 1.0),                  # S = I -> plain cosine
    ([1, 0],    [0, 1],    I2, 0.0),                  # S = I, orthogonal
    ([1, 1],    [1, 0],    I2, 0.7071067811865475),   # S = I, 45 degrees
    ([1, 0],    [0, 1],    S2, 0.5),                  # example 1: no shared features
    ([1, 0],    [1, 0],    S2, 1.0),                  # a vs itself -> 1.0
    ([1, 0],    [-1, 0],   S2, -1.0),                 # opposite
    ([0, 0],    [1, 2],    S2, 0.0),                  # zero vector -> 0.0, NOT nan
    ([0, 0],    [0, 0],    S2, 0.0),                  # both zero
    ([1, 0, 0], [0, 1, 0], S3, 0.9),                  # near-synonyms
    ([1, 0, 0], [0, 0, 1], S3, 0.2),                  # unrelated features
    ([1, 1, 0], [0, 1, 1], S3, 0.7616061053124001),   # partial overlap
]

print("soft_cosine_similarity")
for a, b, S, want in cases:
    print(f"  {str(a):<11} {str(b):<11} -> {check(sol.soft_cosine_similarity(a, b, S), want, 1e-12)}")

print("\nsoft_cosine_cholesky")
for a, b, S, want in cases:
    print(f"  {str(a):<11} {str(b):<11} -> {check(sol.soft_cosine_cholesky(a, b, S), want, 1e-12)}")

# S = I must reproduce plain cosine exactly, on random vectors not just the 3 above
rng = np.random.default_rng(7)
try:
    V = rng.normal(size=(20, 6))
    d = [abs(sol.soft_cosine_similarity(V[i], V[j], np.eye(6)) - sol.cosine_similarity(V[i], V[j]))
         for i in range(20) for j in range(20)]
    print(f"\nS = I matches plain cosine on 400 pairs : {max(d) < 1e-12}")

    # the metric must be symmetric, and self-similarity must be exactly 1
    sym = [abs(sol.soft_cosine_similarity(V[i][:3], V[j][:3], S3)
               - sol.soft_cosine_similarity(V[j][:3], V[i][:3], S3))
           for i in range(20) for j in range(20)]
    print(f"symmetric in a and b                    : {max(sym) < 1e-12}")
    print(f"soft_cosine(a, a) == 1                  : "
          f"{all(abs(sol.soft_cosine_similarity(v[:3], v[:3], S3) - 1) < 1e-12 for v in V)}")

    # PSD guarantees the result stays inside [-1, 1] - check it never escapes
    vals = [sol.soft_cosine_similarity(V[i][:3], V[j][:3], S3) for i in range(20) for j in range(20)]
    print(f"stays within [-1, 1]                    : {max(abs(v) for v in vals) <= 1 + 1e-9}")

    M = np.array([[2.0, 0.0, 0.0],    # scaled copy of a       -> 1.0
                  [0.0, 1.0, 0.0],    # the near-synonym       -> 0.9
                  [0.0, 0.0, 1.0],    # unrelated feature      -> 0.2
                  [0.0, 0.0, 0.0]])   # zero row               -> 0.0
    print("matrix version:", sol.soft_cosine_matrix([1, 0, 0], M, S3), " want [1. 0.9 0.2 0. ]")
except (TypeError, ValueError) as e:
    print(f"\n(finish the functions above to run the cross-checks - {e})")

# the non-PSD case: symmetric, unit diagonal, and still not a valid similarity matrix
BAD = np.array([[1.0, 2.0],
                [2.0, 1.0]])
print("\nBAD eigenvalues:", np.linalg.eigvalsh(BAD), "<- one is negative, so BAD is not PSD")
print("a^T BAD a for a = [1, -1]:", float(np.array([1.0, -1.0]) @ BAD @ np.array([1.0, -1.0])))
print("what does your function do with soft_cosine_similarity([1, -1], [1, 0], BAD)?")

### After it passes: the thing plain cosine cannot see

Three one-word documents over the vocabulary
`["car", "automobile", "truck", "banana"]`. `d1` says *car*, `d2` says
*automobile*, `d3` says *banana*. No two of them share a single word, so plain
cosine gives `0.0` for all three pairs — it cannot tell the synonym pair apart
from the unrelated pair.

Run it and look at what soft cosine returns instead. Then notice where all the
knowledge actually lives: not in the metric, which is four lines, but in `S`. The
metric is only ever as good as the embeddings you built `S` from.

In [ ]:
vocab = ["car", "automobile", "truck", "banana"]

# S from word embeddings: car ~ automobile are near-synonyms, truck is related,
# banana has nothing to do with any of them
S = np.array([
    [1.00, 0.95, 0.70, 0.05],   # car
    [0.95, 1.00, 0.68, 0.05],   # automobile
    [0.70, 0.68, 1.00, 0.08],   # truck
    [0.05, 0.05, 0.08, 1.00],   # banana
])

d1 = np.array([1.0, 0.0, 0.0, 0.0])   # "car"
d2 = np.array([0.0, 1.0, 0.0, 0.0])   # "automobile"
d3 = np.array([0.0, 0.0, 0.0, 1.0])   # "banana"

pairs = [("car", "automobile", d1, d2),
         ("car", "truck",      d1, np.array([0.0, 0.0, 1.0, 0.0])),
         ("car", "banana",     d1, d3)]

def fmt(v):
    return f"{v:>14.4f}" if isinstance(v, float) else f"{'n/a':>14}"

print(f"{'pair':<24}{'cosine':>10}{'soft cosine':>14}")
for x, y, u, v in pairs:
    print(f"{x + ' vs ' + y:<24}{sol.cosine_similarity(u, v):>10.4f}{fmt(sol.soft_cosine_similarity(u, v, S))}")

print("\nplain cosine says all three pairs are equally unrelated. they are not.")
print("every bit of that judgement came out of S, not out of the formula.")